# 03 ML Pipeline: Customer Tribe Discovery

This notebook starts from the prepared parquet outputs created by Notebook 02. It does not repeat raw loading, cleaning, or exploratory data quality work from Notebooks 01 and 02.

The goal is to discover product-first customer tribes from purchase behavior. The client hypothesis is roughly 10-15 tribes, but that range is not used as a modeling constraint. The final recommendation is selected from metrics, stability-ready diagnostics, cluster balance, product/sector lift interpretability, and business usefulness.

## Experiment Workbench

Optional hyperparameter experiments now live in `04_experiment_sandbox.ipynb`. Use that notebook to test variants, promote the winning settings into YAML, then rerun this official pipeline notebook to produce clean artifacts.


## Stage 0: Load Prepared Data

Notebook 03 consumes `df_combined.parquet`, validates the fields required for ML, and merges product metadata only if the prepared file does not already contain it.

In [1]:
import os

# Mode toggle: this cell is authoritative, even if src.config was imported earlier in the kernel.
RUN_MODE = "dev"  # change to "prod" for the full production pipeline
os.environ["CARREFOUR_MODE"] = RUN_MODE.strip().lower()

from IPython.display import Image, Markdown, display

from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "src").is_dir():
        project_root = candidate
        break
else:
    project_root = Path.cwd().resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import configure_mode
from src.data_loader import load_prepared_transactions, peek
from src.utils import set_global_seed

previous_mode = globals().get("MODE")
CONFIG = configure_mode(RUN_MODE)
MODE = CONFIG.mode
DATA_PROCESSED = CONFIG.data_processed
MODELS = CONFIG.models
OUTPUTS = CONFIG.outputs

if previous_mode and previous_mode != MODE:
    for stale_name in [
        "transactions",
        "basket_path",
        "item2vec_model",
        "product_embeddings_path",
        "embedding_validation_csv",
        "embedding_validation_md",
        "customer_embeddings_path",
        "behavior_path",
        "feature_sets",
        "model_suite",
        "stage6_diagnostics",
        "profile_paths",
        "comparison_path",
        "selected",
        "selected_assignment_path",
        "selected_profile_path",
    ]:
        globals().pop(stale_name, None)

set_global_seed(CONFIG.random_seed)
CONFIG.ensure_directories()

transactions = load_prepared_transactions(cfg=CONFIG)
print(f"Run mode: {MODE}")
print(f"Data path: {DATA_PROCESSED}")
print(f"Model path: {MODELS}")
print(f"Output path: {OUTPUTS}")
peek(transactions, 3)

from src.visualization import plot_prepared_data_overview

stage0_figure = plot_prepared_data_overview(transactions, cfg=CONFIG)
display(Image(filename=str(stage0_figure)))


Run mode: dev
Data path: C:\Users\rothl\Desktop\IE Capstone\carrefour_capstone\data\dev
Model path: C:\Users\rothl\Desktop\IE Capstone\carrefour_capstone\outputs\dev\models
Output path: C:\Users\rothl\Desktop\IE Capstone\carrefour_capstone\outputs\dev


: 

## Stage 1: Basket Construction

Each ticket is treated as a basket sentence and each product id is a token. Products are not repeated by quantity unless `baskets.repeat_product_by_quantity` is enabled in the config.

In [ ]:
from IPython.display import Image

from src.basket_builder import basket_summary, build_basket_sentences
from src.visualization import plot_basket_summary

basket_path = build_basket_sentences(transactions=transactions, cfg=CONFIG)
print(basket_path)
basket_summary(basket_path)
stage1_figure = plot_basket_summary(basket_path, cfg=CONFIG)
display(Image(filename=str(stage1_figure)))


## Stage 2: Item2Vec Product Embeddings

The Word2Vec model learns product proximity from basket co-occurrence. These product embeddings are the core signal used to represent customers.

The training function prints the active hyperparameters and one progress line per epoch. If a cached model already exists, it prints the cache details instead; pass `force=True` to retrain.

In [ ]:
from IPython.display import Image

from src.item2vec import save_product_embeddings, train_item2vec
from src.visualization import plot_product_embedding_diagnostics

item2vec_model = train_item2vec(basket_path, cfg=CONFIG, verbose=True)
product_embeddings_path = save_product_embeddings(item2vec_model, cfg=CONFIG)
print(product_embeddings_path)
stage2_figure = plot_product_embedding_diagnostics(product_embeddings_path, cfg=CONFIG)
display(Image(filename=str(stage2_figure)))


## Stage 3: Product Embedding Validation

Before clustering customers, the nearest-neighbor report checks whether embeddings capture meaningful substitutes, complements, or shared basket missions.

In [ ]:
from IPython.display import Image

from src.embedding_validation import validate_product_embeddings
from src.visualization import plot_embedding_validation_summary

embedding_validation_csv, embedding_validation_md = validate_product_embeddings(
    product_embeddings_path,
    transactions=transactions,
    cfg=CONFIG,
)
print(embedding_validation_csv)
print(embedding_validation_md)
stage3_figure = plot_embedding_validation_summary(embedding_validation_csv, cfg=CONFIG)
display(Image(filename=str(stage3_figure)))


## Stage 4: Customer Embeddings

Customer vectors are weighted means of product embeddings. The official weighting uses product quantities (`unidades`) only, so clustering is driven by products and purchase intensity rather than spend. IDF-downweighted variants are tested only in the experiment sandbox before any promotion to YAML.

In [ ]:
from IPython.display import Image

from src.customer_embeddings import build_customer_embeddings
from src.visualization import plot_customer_embedding_diagnostics

customer_embeddings_path = build_customer_embeddings(
    product_embeddings_path,
    transactions=transactions,
    cfg=CONFIG,
)
print(customer_embeddings_path)
stage4_figure = plot_customer_embedding_diagnostics(customer_embeddings_path, cfg=CONFIG)
display(Image(filename=str(stage4_figure)))


## Stage 5: Behavioral Features

Behavioral features are stored separately so the analysis can compare embeddings-only segmentation against embeddings plus behavior without allowing KPIs to silently dominate the product signal.

In [ ]:
from IPython.display import Image

from src.feature_engineering import build_behavioral_features, build_feature_set
from src.visualization import plot_behavioral_feature_summary, plot_feature_set_summary

behavior_path = build_behavioral_features(transactions=transactions, cfg=CONFIG)
feature_set_a = build_feature_set(customer_embeddings_path, variant="embeddings_only", cfg=CONFIG)
feature_set_b = build_feature_set(
    customer_embeddings_path,
    behavior_path=behavior_path,
    variant="embeddings_behavior",
    cfg=CONFIG,
)
feature_sets = {
    "embeddings_only": feature_set_a,
    "embeddings_behavior": feature_set_b,
}
print(behavior_path)
print(feature_sets)
stage5_behavior_figure = plot_behavioral_feature_summary(behavior_path, cfg=CONFIG)
stage5_feature_set_figure = plot_feature_set_summary(feature_sets, cfg=CONFIG)
display(Image(filename=str(stage5_behavior_figure)))
display(Image(filename=str(stage5_feature_set_figure)))


## Stage 6: Candidate Model Comparison

The official pipeline compares algorithm families without forcing a target tribe count. Broader hyperparameter sweeps belong in `04_experiment_sandbox.ipynb`; promote only evidence-backed settings into YAML before running this notebook.

- `model_a_gmm`: Raw customer embeddings to Gaussian Mixture Model, searched across the configured `gmm.components_min` to `gmm.components_max` range.
- `model_b_umap_hdbscan`: UMAP customer manifold to HDBSCAN using the active YAML `umap` and `hdbscan` settings.
- `model_c_pca_kmeans`: PCA representation to MiniBatchKMeans, searched across the configured `kmeans.k_min` to `kmeans.k_max` range.

UMAP is treated as a candidate clustering representation, not as an automatic winner. Its value must be proven by metrics, stability, and product-lift interpretability.

In [ ]:
import polars as pl
from IPython.display import Image, Markdown, display

from src.model_selection import build_candidate_model_diagnostics, run_candidate_model_suite
from src.visualization import plot_stage6_model_diagnostics

if "feature_sets" not in globals():
    feature_set_outputs = CONFIG.get("feature_sets.outputs", {})
    feature_sets = {
        name: CONFIG.outputs / "features" / filename
        for name, filename in feature_set_outputs.items()
    }
    missing_feature_sets = [path for path in feature_sets.values() if not path.exists()]
    if missing_feature_sets:
        raise FileNotFoundError(
            "Stage 6 needs the Stage 5 feature-set parquet files. "
            f"Missing: {missing_feature_sets}. Run Stage 5 first."
        )

selection_feature_set = CONFIG.get("modeling.feature_set_for_selection", "embeddings_only")
model_suite = run_candidate_model_suite(feature_sets[selection_feature_set], cfg=CONFIG)
stage6_diagnostics = build_candidate_model_diagnostics(model_suite, cfg=CONFIG)
stage6_figure = plot_stage6_model_diagnostics(stage6_diagnostics["parquet"], cfg=CONFIG)

display(Markdown(f"""
### Stage 6 Diagnostics

- Diagnostics parquet: `{stage6_diagnostics['parquet']}`
- Diagnostics summary CSV: `{stage6_diagnostics['summary_csv']}`
- Diagnostics summary Markdown: `{stage6_diagnostics['summary_md']}`
- Diagnostics figure: `{stage6_figure}`
"""))
display(Image(filename=str(stage6_figure)))

pl.read_parquet(stage6_diagnostics["parquet"]).select([
    "stage6_rank",
    "model_name",
    "algorithm_name",
    "model_variant",
    "cluster_count",
    "coverage_adjusted_silhouette",
    "silhouette",
    "davies_bouldin",
    "noise_pct",
    "passes_quality_gate",
]).head(25)

## Stage 7: Cluster Validity and Stability

This section is a guardrail before interpretation. It reviews cluster count, noise, balance, and a lightweight perturbation check that asks whether labels remain recoverable from the original customer-product feature geometry. This does not replace a full repeated-seed stability study, but it makes instability visible before tribe profiles are interpreted.

In [ ]:
from src.cluster_validation import build_cluster_validity_stability_report

cluster_validity = build_cluster_validity_stability_report(
    model_suite,
    feature_sets[selection_feature_set],
    cfg=CONFIG,
)

display(Markdown(f"""
### Cluster Validity and Stability

- Validity/stability parquet: `{cluster_validity['parquet']}`
- Validity/stability summary CSV: `{cluster_validity['summary_csv']}`
- Validity/stability summary Markdown: `{cluster_validity['summary_md']}`
"""))

pl.read_parquet(cluster_validity["parquet"]).select([
    "candidate_id",
    "cluster_count",
    "noise_pct",
    "cluster_size_cv",
    "jitter_ari_mean",
    "jitter_ari_std",
    "jitter_label_recovery_accuracy_mean",
    "validity_note",
])

## Stage 8: Tribe Profiling and Interpretability

Every candidate solution is profiled using product lift and sector lift. Raw product popularity is not enough because staples tend to dominate all customers.

In [ ]:
from src.profiling import profile_tribes

profile_paths = {}
for candidate_key, assignment_path in model_suite["assignment_paths"].items():
    profile_paths[candidate_key] = profile_tribes(
        assignment_path,
        transactions=transactions,
        behavior_path=behavior_path,
        cfg=CONFIG,
    )
profile_paths

## Stage 9: Final Model Selection, Exports, and Figures

The comparison table ranks candidates using empirical metrics, cluster balance, assignment confidence where available, HDBSCAN noise share where relevant, and product/sector lift interpretability.

In [ ]:
from src.exports import (
    build_model_comparison,
    export_final_assignments,
    export_final_profiles,
    selected_model,
    write_decision_log,
)
from IPython.display import Image

from src.visualization import (
    build_2d_projection_figures,
    plot_candidate_umap_grid,
    plot_cluster_sizes,
    plot_model_comparison,
    plot_top_lifts,
)

comparison_path = build_model_comparison(
    model_suite["candidate_results"],
    profile_paths=profile_paths,
    cfg=CONFIG,
)
selected = selected_model(comparison_path)
selected_key = f"{selected['model_name']}::{selected['model_variant']}"
selected_assignment_path = model_suite["assignment_paths"][selected_key]
selected_profile_path = profile_paths[selected_key]

assignment_export = export_final_assignments(selected_assignment_path, cfg=CONFIG)
profile_export = export_final_profiles(selected_profile_path, cfg=CONFIG)
decision_log_path = write_decision_log(comparison_path, selected_profile_path, cfg=CONFIG)

figures = {
    "model_comparison": plot_model_comparison(comparison_path, cfg=CONFIG),
    "candidate_umap_grid": plot_candidate_umap_grid(
        feature_sets[selection_feature_set],
        model_suite["assignment_paths"],
        cfg=CONFIG,
    ),
    "cluster_sizes": plot_cluster_sizes(selected_assignment_path, cfg=CONFIG),
    "projection": build_2d_projection_figures(feature_sets[selection_feature_set], selected_assignment_path, cfg=CONFIG),
    "lift_plots": plot_top_lifts(selected_profile_path, cfg=CONFIG),
}

display(Image(filename=str(figures["model_comparison"])))
display(Image(filename=str(figures["candidate_umap_grid"])))
if "umap" in figures["projection"]:
    display(Image(filename=str(figures["projection"]["umap"])))
display(Image(filename=str(figures["cluster_sizes"])))

display(Markdown(f"""
### Selected Solution

- Model: **{selected['model_name']} ({selected['model_variant']})**
- Tribes: **{selected['cluster_count']}**
- Comparison table: `{comparison_path}`
- Assignments: `{assignment_export}`
- Tribe profiles: `{profile_export}`
- Decision log: `{decision_log_path}`
- Figures folder: `{CONFIG.figures}`
"""))

## Decision Log: Final Tribe Model Selection

This final section is generated after model evaluation and profiling. It records which model was selected, how many tribes were selected, whether that result agrees with the 10-15 tribe client hypothesis, why the selected model won, why alternatives were rejected, what evidence supports the recommendation, remaining limitations, and next production rollout steps.

In [ ]:
from pathlib import Path

display(Markdown(Path(decision_log_path).read_text(encoding="utf-8")))